# 37 — Decoder Stack and Language-Model Head

**Master syllabus coverage:** V2 5.13–5.14

## Why this matters

This is the first complete decoder forward graph: IDs become embeddings, blocks exchange and transform information, and the LM head emits one vocabulary distribution per position.

## Learning objectives

- Assemble embeddings, repeated blocks, final norm, and LM head.
- Compute `[B,T,V]` logits and next-token loss.
- Implement true input/output weight tying.
- Account for parameters and verify all gradients are finite.

Work through the notebook in order. Predict shapes and results before running each code cell. An assertion failure is part of the lesson: read it before changing code.


## 1. Stack blocks into a decoder-only language model

Token and position embeddings initialize `[B,T,C]`. `L` decoder blocks repeatedly update
it. A final normalization produces states for an LM head mapping `C → V`; logits are
`[B,T,V]`. The model predicts each next token in parallel during training because a causal
mask prevents forbidden context.


In [ ]:
import math
from dataclasses import dataclass
import torch
import torch.nn.functional as F

@dataclass(frozen=True)
class Config:
    vocab_size: int = 23
    context: int = 12
    width: int = 32
    heads: int = 4
    layers: int = 2
    dropout: float = 0.0

class Attention(torch.nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.width % config.heads == 0
        self.heads = config.heads
        self.head_dim = config.width // config.heads
        self.qkv = torch.nn.Linear(config.width, 3 * config.width)
        self.out = torch.nn.Linear(config.width, config.width)

    def forward(self, x):
        B, T, C = x.shape
        q, k, v = self.qkv(x).chunk(3, dim=-1)
        def split(z): return z.view(B, T, self.heads, self.head_dim).transpose(1, 2)
        q, k, v = map(split, (q, k, v))
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        return self.out(y.transpose(1, 2).contiguous().view(B, T, C))

class Block(torch.nn.Module):
    def __init__(self, config):
        super().__init__()
        self.norm1 = torch.nn.LayerNorm(config.width)
        self.attention = Attention(config)
        self.norm2 = torch.nn.LayerNorm(config.width)
        self.mlp = torch.nn.Sequential(
            torch.nn.Linear(config.width, 4 * config.width), torch.nn.GELU(),
            torch.nn.Linear(4 * config.width, config.width),
        )
    def forward(self, x):
        x = x + self.attention(self.norm1(x))
        return x + self.mlp(self.norm2(x))


In [ ]:
class TinyDecoder(torch.nn.Module):
    def __init__(self, config: Config, tie_weights: bool = True):
        super().__init__()
        self.config = config
        self.token_embedding = torch.nn.Embedding(config.vocab_size, config.width)
        self.position_embedding = torch.nn.Embedding(config.context, config.width)
        self.blocks = torch.nn.ModuleList([Block(config) for _ in range(config.layers)])
        self.final_norm = torch.nn.LayerNorm(config.width)
        self.lm_head = torch.nn.Linear(config.width, config.vocab_size, bias=False)
        if tie_weights:
            self.lm_head.weight = self.token_embedding.weight

    def forward(self, token_ids, targets=None):
        B, T = token_ids.shape
        if T > self.config.context:
            raise ValueError(f"sequence length {T} exceeds context {self.config.context}")
        positions = torch.arange(T, device=token_ids.device)
        x = self.token_embedding(token_ids) + self.position_embedding(positions)[None]
        for block in self.blocks:
            x = block(x)
        logits = self.lm_head(self.final_norm(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.reshape(-1, logits.shape[-1]), targets.reshape(-1))
        return logits, loss

torch.manual_seed(42)
config = Config()
model = TinyDecoder(config)
token_ids = torch.randint(0, config.vocab_size, (3, 8))
targets = torch.randint(0, config.vocab_size, (3, 8))
logits, loss = model(token_ids, targets)
print("logits:", logits.shape, "loss:", loss.item())
assert logits.shape == (3, 8, config.vocab_size)


## 2. Weight tying

Input embeddings map token IDs into features; the LM head compares features with output
token rows. Tying uses the same `[V,C]` parameter for both, reducing parameter count and
often improving statistical efficiency. Assignment must preserve true parameter identity,
not copy values once.


In [ ]:
assert model.token_embedding.weight is model.lm_head.weight
untied = TinyDecoder(config, tie_weights=False)
tied_params = sum(p.numel() for p in model.parameters())
untied_params = sum(p.numel() for p in untied.parameters())
print("tied:", tied_params, "untied:", untied_params, "saved:", untied_params - tied_params)
assert untied_params - tied_params == config.vocab_size * config.width


## 3. Initialization and parameter accounting

Parameter count is a testable architectural property. Embeddings scale with `V×C` and
`T×C`; each classic block is dominated by about `12C²` weights (4C² attention and 8C²
MLP). Exact counts include biases and normalization parameters.


In [ ]:
for name, parameter in model.named_parameters():
    assert torch.isfinite(parameter).all()
    print(f"{name:40} {tuple(parameter.shape)!s:16} {parameter.numel():6}")
print("total:", tied_params)

loss.backward()
assert all(parameter.grad is not None and torch.isfinite(parameter.grad).all()
           for parameter in model.parameters())
print("all trainable parameters received finite gradients")


## Failure modes to recognize

- **Context overflow:** position embeddings or masks do not cover the requested sequence.
- **Copied rather than tied weights:** embedding and head diverge after one update.
- **Missing final norm:** architecture no longer matches the intended pre-norm design.
- **Wrong flatten order:** logits and targets refer to different positions.

These are diagnostic patterns, not trivia. When a future model fails, reduce the problem to the smallest example in this notebook and test the relevant invariant.


## Deliberate practice

1. Derive an exact parameter-count formula and verify it for three configurations.
2. Add an `ignore_index` for padded targets and test that padding changes neither loss nor gradients.
3. Test end-to-end causality by changing suffix token IDs and comparing prefix logits.

Do not merely rerun the provided cells. Make a prediction, change one variable, and write down why the result did or did not match your prediction.

## Exit ticket

**You are ready to continue when:** you can construct the full logits graph, explain every state-dict tensor, and prove parameter, shape, gradient, and causality invariants.

**Next:** 38 — Transformer complexity and failure analysis.
